In [56]:
from dotenv import load_dotenv
# loads the environment variables from a .env file into the system's environment variables
# This contains the API keys and other sensitive information that should not be hardcoded into the codebase.
# The API keys are used to authenticate the agent with the language model provider and other services.

load_dotenv()

True

In [57]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()
#TavilyClient is a client for the Tavily API, which allows you to search the web for information. It is used in the web_search tool to perform web searches.

# Decorator to register the function as a tool that can be used by the agent. The tool takes a query string as input and returns a dictionary containing the search results from the Tavily API.
# tool decorator is used to register the function as a tool that can be used by the agent. The tool takes a query string as input and returns a dictionary containing the search results from the Tavily API.
# Tools are functions that can be called by the agent to perform specific tasks, such as searching the web, retrieving data from a database, or performing calculations. The agent can use these tools to augment its capabilities and provide more accurate and relevant responses to user queries.
@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""
# LangSmith traces the execution of the tool, allowing you to see how the tool was called and what it returned. This can be useful for debugging and understanding how the agent is using the tool.
# My LangSmith API key is stored in the .env file and is loaded into the system's environment variables using the load_dotenv() function. This allows the agent to authenticate with the LangSmith API and use its tracing capabilities.
# LangSmith Tracing for this is : https://smith.langchain.com/o/a19995d9-0fdf-402c-a3ac-be4a34b95eb1/projects/p/e6566967-fb06-4a35-b245-0021ef30badb
    return tavily_client.search(query)

In [59]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [60]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [61]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [62]:
print(uploader.value)

({'name': 'Messy_LeftOver.png', 'type': 'image/png', 'size': 87042, 'content': <memory at 0x130eee380>, 'last_modified': datetime.datetime(2026, 10, 3, 23, 28, 51, 691000, tzinfo=datetime.timezone.utc)},)


In [63]:
import base64
import subprocess
import tempfile
from pathlib import Path

uploaded_file = uploader.value[0]
content_mv = uploaded_file["content"]
img_bytes = bytes(content_mv)
image_mime_type = "image/png"

if img_bytes[4:12] == b"ftypavif":
    with tempfile.TemporaryDirectory() as temp_dir:
        source_path = Path(temp_dir) / "upload.avif"
        output_path = Path(temp_dir) / "upload.jpg"
        source_path.write_bytes(img_bytes)
        subprocess.run(
            ["sips", "-s", "format", "jpeg", str(source_path), "--out", str(output_path)],
            check=True,
            capture_output=True,
        )
        img_bytes = output_path.read_bytes()
    image_mime_type = "image/jpeg"

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [64]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Nice! Leftover chicken and rice are surprisingly versatile. Here are tasty options you can make, ranging from quick stovetop dinners to a cozy casserole. I’ve noted roughly how long they take and what extra ingredients you’d need.

- Chicken Fried Rice (quick one-pan)
  - Why it fits: Uses both leftover chicken and rice, plus a few pantry veggies.
  - Time: about 15 minutes.
  - Extra: soy sauce, eggs, peas/carrots (or any frozen veggie mix), green onions, sesame oil (optional).
  - Source: https://quickweeknightmeals.com/chicken-fried-rice

- One-Pot Chicken and Rice Soup
  - Why it fits: Cozy, uses chicken stock/broth, and reheats beautifully.
  - Time: about 25–35 minutes.
  - Extra: carrots, celery, onion, garlic, optional lemon juice or parsley for brightness.
  - Source: https://www.saltandlavender.com/easy-chicken-and-rice-soup

- Arroz Con Pollo (one-pot chicken and rice, Latin style)
  - Why it fits: A flavorful, comforting one-pan meal; great with peppers, tomatoes, and spice

In [65]:
from uuid import uuid4

image_config = {"configurable": {"thread_id": str(uuid4())}}
# The image_config dictionary is used to configure the agent's behavior when processing multimodal inputs, such as images. The thread_id is set to a unique value generated by uuid4(), which allows the agent to maintain context across multiple interactions with the user. This is important for providing coherent and relevant responses based on the user's previous inputs and the current context of the conversation.
# correctly identify the image and maintain context across multiple interactions with the user.

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "I have some leftover as shown in the picture. What can I make?"},
    {"type": "image", "base64": img_b64, "mime_type": image_mime_type}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config=image_config
)
#

print(response['messages'][-1].content)

Nice fridge! With eggs, cabbage/greens, lettuce, olives, and fruit you’ve got plenty of fast, tasty options. Here are some ideas that use those leftovers:

- Cabbage and eggs stir-fry (Bắp Cải Xào Trứng)
  - Why: Quick one-pan meal, great for breakfast or dinner.
  - What you’ll do: Sauté shredded cabbage with garlic, season, then scramble in some eggs.
  - Link: https://simplevietnamesefood.com/egg-cabbage-stir-fry-bap-cai-xao-trung

- Cabbage and eggs skillet (classic fried cabbage with eggs)
  - Why: Simple, comforting, uses cabbage and eggs up fast.
  - What you’ll do: Sauté cabbage (and onion/garlic if you have), push to side, scramble eggs, mix together.
  - Link: https://www.joybauer.com/healthy-recipes/cabbage-and-eggs-skillet-quick-easy

- Veggie omelet (greens + whatever you have)
  - Why: Super quick, flexible, great for using greens like spinach/kale along with lettuce.
  - What you’ll do: Sauté greens, whisk eggs, pour over, add any cheese or tomatoes if you have them.
  -

In [66]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'I have some leftover as shown in the picture. What can I make?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='3bf84a31-cb24-4bf0-b7c3-20df5cc767bb'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1379, 'prompt_tokens': 764, 'total_tokens': 2143, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1152, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EV3T9KE86xMIdpVNFcxNrbOHarT55', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1041a-8ca9-7303-b94e-310f6c3be07d-0', tool_calls=[{'name': 'web_search